# 04 - GMAP Evaluation

This notebook evaluates the robustness of RRPR perturbations using
Generalized Morph Attack Potential (GMAP).

Face Recognition Systems:

- AdaFace
- ArcFace
- MagFace
- ElasticFace
- EdgeFace

Datasets:

- FERET
- FRGC

Inputs:

- Embeddings/

Outputs:

- GMAP_Results/
- Table 1

Expected Runtime:

Demo Mode:
- 10-15 minutes

Full Dataset:
- 40-50 minutes

GPU Required:

- No

In [1]:
try:
    from google.colab import drive

    drive.mount("/content/drive")

    print("Google Drive mounted.")

except ImportError:

    print("Running outside Colab. Drive mount skipped.")

Mounted at /content/drive


In [2]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

## Configuration

This section defines:

- Embedding locations
- Datasets
- FRS models
- Perturbation types
- Morph generators

In [3]:
# update accordingly
PROJECT_ROOT      = Path("/content/drive/MyDrive/Free-Cloud/ICPR_Rep")
EMB_ROOT          = PROJECT_ROOT / "Embeddings"
GMAP_RESULTS_ROOT = PROJECT_ROOT / "GMAP_Results"

GMAP_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print(EMB_ROOT)
print(GMAP_RESULTS_ROOT)

/content/drive/MyDrive/Free-Cloud/ICPR_Rep/Embeddings
/content/drive/MyDrive/Free-Cloud/ICPR_Rep/GMAP_Results


In [4]:
MODELS = ["adaface", "arcface", "magface", "elasticface", "edgeface"]

DATASETS = ["FERET", "FRGC"]

PERTURBATIONS = [
    "original",
    "Weighted_Ensemble_Template_PGD",
    "Weighted_Ensemble_Template_DCT_HF",
    "Weighted_Ensemble_Template_DWT_HF",
    "Weighted_Ensemble_Template_BPDA_EOT"
]

MORPH_TYPES = ["greedy", "mipgan2", "ubo"]

## Verification Thresholds

GMAP is computed using the verification thresholds reported in the paper.

Thresholds are provided for:

- FAR = 1%
- FAR = 0.1%

for each FRS.

In [5]:
THRESHOLDS = {
    0.01: {
        "adaface": 0.17550,
        "arcface": 0.23195,
        "magface": 0.25485,
        "elasticface": 0.25485,
        "edgeface": 0.25485
    },
    0.001: {
        "adaface": 0.25006,
        "arcface": 0.32973,
        "magface": 0.34922,
        "elasticface": 0.34922,
        "edgeface": 0.34922
    }
}

In [6]:
def l2_normalize(x):
    norm = np.linalg.norm(x)
    if norm == 0:
        return x
    return x / norm


def cosine_similarity(a, b):
    a = l2_normalize(a)
    b = l2_normalize(b)
    return float(np.dot(a, b))

## Morph Parsing

Morph filenames follow the format:

source1-vs-source2

Example:

00497_0-vs-00695_1

allowing exact source embeddings to be recovered.

In [7]:
def parse_morph_filename(filename):
    base = Path(filename).stem
    left, right = base.split("-vs-")
    return left, right


print(parse_morph_filename("00497_0-vs-00695_1.npy"))

('00497_0', '00695_1')


In [8]:
# Find Source

def find_source_embedding(aligned_root, image_id):
    candidates = list(aligned_root.rglob(f"{image_id}.npy"))
    if len(candidates) == 0:
        return None
    return candidates[0]

## Similarity Computation

For every morph:

cos(source1, morph)

cos(source2, morph)

are computed using L2-normalized embeddings.

In [9]:
def compute_morph_similarities(morph_path, aligned_root):
    src1_id, src2_id = parse_morph_filename(morph_path.name)

    src1_path = find_source_embedding(aligned_root, src1_id)
    src2_path = find_source_embedding(aligned_root, src2_id)

    if src1_path is None or src2_path is None:
        return None

    morph_emb = np.load(morph_path)
    src1_emb  = np.load(src1_path)
    src2_emb  = np.load(src2_path)

    sim1 = cosine_similarity(src1_emb, morph_emb)
    sim2 = cosine_similarity(src2_emb, morph_emb)

    return {"sim_source1": sim1, "sim_source2": sim2}

In [11]:
def build_similarity_dataframe(model, dataset, perturbation, morph_type):
    morph_root   = EMB_ROOT / model / dataset / perturbation / "morph" / morph_type / "test"
    aligned_root = EMB_ROOT / model / dataset / perturbation / "aligned"

    rows = []
    morph_files = list(morph_root.rglob("*.npy"))

    for morph_path in morph_files:
        result = compute_morph_similarities(morph_path, aligned_root)

        if result is None:
            continue

        rows.append({
            "model": model,
            "dataset": dataset,
            "perturbation": perturbation,
            "morph_type": morph_type,
            "sim_source1": result["sim_source1"],
            "sim_source2": result["sim_source2"]
        })

    return pd.DataFrame(rows)

## GMAP Evaluation

A morph attack is considered successful when:

source1 similarity ≥ threshold

AND

source2 similarity ≥ threshold

for the selected FRS.

In [12]:
# GMAP

def compute_gmap(df, threshold):
    if len(df) == 0:
        return np.nan

    success = (df["sim_source1"] >= threshold) & (df["sim_source2"] >= threshold)
    return success.mean() * 100.0

In [13]:
# single FRS

def evaluate_single_frs(model, dataset, perturbation, morph_type):
    df = build_similarity_dataframe(model, dataset, perturbation, morph_type)

    gmap_1  = compute_gmap(df, THRESHOLDS[0.01][model])
    gmap_01 = compute_gmap(df, THRESHOLDS[0.001][model])

    return {"GMAP@1": gmap_1, "GMAP@0.1": gmap_01}

## All-FRS Evaluation

A morph attack is considered successful only when it succeeds against:

- AdaFace
- ArcFace
- MagFace
- ElasticFace
- EdgeFace

simultaneously.

In [14]:
# all frs

def evaluate_all_frs(dataset, perturbation, morph_type):
    dfs = {}

    for model in MODELS:
        dfs[model] = build_similarity_dataframe(model, dataset, perturbation, morph_type)

    n = min(len(df) for df in dfs.values())

    if n == 0:
        return {"GMAP@1": np.nan, "GMAP@0.1": np.nan}

    success_1  = np.ones(n, dtype=bool)
    success_01 = np.ones(n, dtype=bool)

    for model in MODELS:
        df = dfs[model].iloc[:n]

        success_1  &= (df["sim_source1"] >= THRESHOLDS[0.01][model])  & (df["sim_source2"] >= THRESHOLDS[0.01][model])
        success_01 &= (df["sim_source1"] >= THRESHOLDS[0.001][model]) & (df["sim_source2"] >= THRESHOLDS[0.001][model])

    return {"GMAP@1": success_1.mean() * 100, "GMAP@0.1": success_01.mean() * 100}

## Evaluation

This section evaluates:

- Original
- PGD
- DCT-HF
- DWT-HF
- BPDA-EOT

for:

- FERET
- FRGC

using:

- Greedy
- MIPGAN-II
- UBO

In [15]:
results = []

for dataset in DATASETS:
    print("\n" + "=" * 80)
    print(dataset)
    print("=" * 80)

    for perturbation in PERTURBATIONS:
        print(f"\n{perturbation}")

        for morph_type in MORPH_TYPES:

            # Individual FRS
            for model in MODELS:
                metrics = evaluate_single_frs(model, dataset, perturbation, morph_type)
                results.append({
                    "dataset": dataset,
                    "perturbation": perturbation,
                    "frs": model,
                    "morph_type": morph_type,
                    **metrics
                })

            # All_FRS
            metrics = evaluate_all_frs(dataset, perturbation, morph_type)
            results.append({
                "dataset": dataset,
                "perturbation": perturbation,
                "frs": "All_FRS",
                "morph_type": morph_type,
                **metrics
            })


FERET

original

Weighted_Ensemble_Template_PGD

Weighted_Ensemble_Template_DCT_HF

Weighted_Ensemble_Template_DWT_HF

Weighted_Ensemble_Template_BPDA_EOT

FRGC

original

Weighted_Ensemble_Template_PGD

Weighted_Ensemble_Template_DCT_HF

Weighted_Ensemble_Template_DWT_HF

Weighted_Ensemble_Template_BPDA_EOT


In [16]:
results_df = pd.DataFrame(results)

raw_csv = GMAP_RESULTS_ROOT / "gmap_raw_results.csv"
results_df.to_csv(raw_csv, index=False)

print(raw_csv)
display(results_df.head())

/content/drive/MyDrive/Free-Cloud/ICPR_Rep/GMAP_Results/gmap_raw_results.csv


,dataset,perturbation,frs,morph_type,GMAP@1,GMAP@0.1
0,FERET,original,adaface,greedy,100.0,100.0
1,FERET,original,arcface,greedy,100.0,100.0
2,FERET,original,magface,greedy,100.0,100.0
3,FERET,original,elasticface,greedy,100.0,100.0
4,FERET,original,edgeface,greedy,100.0,100.0


In [17]:
from scipy.stats import t

def mean_ci_95(values):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    n = len(values)

    if n == 0:
        return np.nan, np.nan

    mean = np.mean(values)

    if n == 1:
        return mean, 0.0

    sem = np.std(values, ddof=1) / np.sqrt(n)
    ci = t.ppf(0.975, n - 1) * sem

    return mean, ci

## Aggregation

Results are aggregated across:

- Greedy
- MIPGAN-II
- UBO

and reported as:

mean ± 95% CI

In [18]:
aggregated_rows = []

for keys, group_df in results_df.groupby(["dataset", "perturbation", "frs"]):
    dataset, perturbation, frs = keys

    row = {"dataset": dataset, "perturbation": perturbation, "frs": frs}

    mean_1,  ci_1  = mean_ci_95(group_df["GMAP@1"])
    mean_01, ci_01 = mean_ci_95(group_df["GMAP@0.1"])

    row["GMAP@1_mean"]   = mean_1
    row["GMAP@1_ci"]     = ci_1
    row["GMAP@0.1_mean"] = mean_01
    row["GMAP@0.1_ci"]   = ci_01

    aggregated_rows.append(row)

aggregated_df = pd.DataFrame(aggregated_rows)
aggregated_df = aggregated_df.sort_values(["dataset", "perturbation", "frs"]).reset_index(drop=True)

display(aggregated_df.head())

,dataset,perturbation,frs,GMAP@1_mean,GMAP@1_ci,GMAP@0.1_mean,GMAP@0.1_ci
0,FERET,Weighted_Ensemble_Template_BPDA_EOT,All_FRS,62.962963,114.914334,25.925926,57.457167
1,FERET,Weighted_Ensemble_Template_BPDA_EOT,adaface,83.333333,60.156140,64.814815,115.740076
2,FERET,Weighted_Ensemble_Template_BPDA_EOT,arcface,87.037037,55.775128,59.259259,128.230819
3,FERET,Weighted_Ensemble_Template_BPDA_EOT,edgeface,62.962963,114.914334,42.592593,91.890022
4,FERET,Weighted_Ensemble_Template_BPDA_EOT,elasticface,70.370370,104.497707,50.000000,104.193491


In [19]:
agg_csv = GMAP_RESULTS_ROOT / "gmap_mean_ci_results.csv"
aggregated_df.to_csv(agg_csv, index=False)
print(agg_csv)

/content/drive/MyDrive/Free-Cloud/ICPR_Rep/GMAP_Results/gmap_mean_ci_results.csv


In [20]:
def format_mean_ci(mean, ci, decimals=2):
    if np.isnan(mean):
        return "-"
    return f"{mean:.2f} ± {ci:.2f}"

## Paper Table Generation

This section generates the final GMAP table reported in the paper.

In [21]:
# Feret

FERET_ORDER = ["adaface", "magface", "arcface", "elasticface", "edgeface", "All_FRS"]

PERT_ORDER = [
    "original",
    "Weighted_Ensemble_Template_DCT_HF",
    "Weighted_Ensemble_Template_DWT_HF",
    "Weighted_Ensemble_Template_BPDA_EOT",
    "Weighted_Ensemble_Template_PGD"
]

rows = []

for perturbation in PERT_ORDER:
    for frs in FERET_ORDER:
        subset = aggregated_df[
            (aggregated_df["dataset"] == "FERET") &
            (aggregated_df["perturbation"] == perturbation) &
            (aggregated_df["frs"] == frs)
        ]

        if len(subset) == 0:
            continue

        rows.append({
            "Perturbation": perturbation,
            "FRS": frs,
            "GMAP-MA@1%":   format_mean_ci(subset.iloc[0]["GMAP@1_mean"],   subset.iloc[0]["GMAP@1_ci"]),
            "GMAP-MA@0.1%": format_mean_ci(subset.iloc[0]["GMAP@0.1_mean"], subset.iloc[0]["GMAP@0.1_ci"])
        })

feret_table = pd.DataFrame(rows)
display(feret_table)

,Perturbation,FRS,GMAP-MA@1%,GMAP-MA@0.1%
0,original,adaface,100.00 ± 0.00,81.48 ± 79.68
1,original,magface,88.89 ± 47.81,72.22 ± 96.61
2,original,arcface,94.44 ± 23.90,70.37 ± 104.50
3,original,elasticface,85.19 ± 63.74,62.96 ± 105.41
4,original,edgeface,74.07 ± 99.84,57.41 ± 118.99
5,original,All_FRS,72.22 ± 107.79,46.30 ± 125.22
6,Weighted_Ensemble_Template_DCT_HF,adaface,88.89 ± 47.81,74.07 ± 111.55
7,Weighted_Ensemble_Template_DCT_HF,magface,87.04 ± 55.78,62.96 ± 125.22
8,Weighted_Ensemble_Template_DCT_HF,arcface,79.63 ± 76.01,62.96 ± 136.16
9,Weighted_Ensemble_Template_DCT_HF,elasticface,74.07 ± 99.84,57.41 ± 128.23


In [22]:
# FRGC
rows = []

for perturbation in PERT_ORDER:
    for frs in FERET_ORDER:
        subset = aggregated_df[
            (aggregated_df["dataset"] == "FRGC") &
            (aggregated_df["perturbation"] == perturbation) &
            (aggregated_df["frs"] == frs)
        ]

        if len(subset) == 0:
            continue

        rows.append({
            "Perturbation": perturbation,
            "FRS": frs,
            "GMAP-MA@1%":   format_mean_ci(subset.iloc[0]["GMAP@1_mean"],   subset.iloc[0]["GMAP@1_ci"]),
            "GMAP-MA@0.1%": format_mean_ci(subset.iloc[0]["GMAP@0.1_mean"], subset.iloc[0]["GMAP@0.1_ci"])
        })

frgc_table = pd.DataFrame(rows)
display(frgc_table)

,Perturbation,FRS,GMAP-MA@1%,GMAP-MA@0.1%
0,original,adaface,100.00 ± 0.00,98.33 ± 7.17
1,original,magface,100.00 ± 0.00,91.67 ± 18.97
2,original,arcface,98.33 ± 7.17,85.00 ± 44.78
3,original,elasticface,98.33 ± 7.17,86.67 ± 31.26
4,original,edgeface,91.67 ± 25.86,70.00 ± 64.54
5,original,All_FRS,90.00 ± 24.84,60.00 ± 89.57
6,Weighted_Ensemble_Template_DCT_HF,adaface,100.00 ± 0.00,98.33 ± 7.17
7,Weighted_Ensemble_Template_DCT_HF,magface,100.00 ± 0.00,96.67 ± 7.17
8,Weighted_Ensemble_Template_DCT_HF,arcface,96.67 ± 14.34,83.33 ± 43.62
9,Weighted_Ensemble_Template_DCT_HF,elasticface,98.33 ± 7.17,78.33 ± 50.20


In [23]:
feret_csv = GMAP_RESULTS_ROOT / "table1_feret.csv"
frgc_csv  = GMAP_RESULTS_ROOT / "table1_frgc.csv"

feret_table.to_csv(feret_csv, index=False)
frgc_table.to_csv(frgc_csv, index=False)

print(feret_csv)
print(frgc_csv)

/content/drive/MyDrive/Free-Cloud/ICPR_Rep/GMAP_Results/table1_feret.csv
/content/drive/MyDrive/Free-Cloud/ICPR_Rep/GMAP_Results/table1_frgc.csv


## Generated Outputs
``` text
GMAP_Results/

├── gmap_raw_results.csv
├── gmap_mean_ci_results.csv
├── table1_feret.csv
└── table1_frgc.csv
```
These files reproduce the GMAP results reported in the paper.